# Diagnóstico inicial reproducible (fase 1)

Este cuaderno descarga datos **pseudonimizados** desde GitHub y genera automáticamente las tablas y gráficos del diagnóstico inicial. Para ejecutar: configurar `PROPIETARIO_GITHUB` y seleccionar **Entorno de ejecución → Ejecutar todo**. No requiere Google Drive.

**Publicación condicionada:** los datos mantienen fechas y cantidades originales; no publicar sin revisar el alcance de la autorización y el riesgo de reidentificación.


In [ ]:
# Modo público: cargar automáticamente los CSV del repositorio GitHub.
# Configura PROPIETARIO_GITHUB con el propietario REAL de tu repositorio.
import pandas as pd
import matplotlib.pyplot as plt

PROPIETARIO_GITHUB = "REEMPLAZAR_USUARIO"  # modificar una sola vez antes de publicar
REPOSITORIO = "tesis-bi-cbn"
RAMA = "principal"
if PROPIETARIO_GITHUB == "REEMPLAZAR_USUARIO":
    raise ValueError("Escribe el nombre real del propietario de GitHub en PROPIETARIO_GITHUB")

base_url = f"https://raw.githubusercontent.com/{PROPIETARIO_GITHUB}/{REPOSITORIO}/{RAMA}/datos/muestra"
url_datos = base_url + "/datos_diagnostico_pseudonimizados.csv"
url_catalogo = base_url + "/catalogo_pseudonimizado.csv"

df = pd.read_csv(url_datos, dtype=str, encoding="utf-8-sig")
maestro = pd.read_csv(url_catalogo, dtype=str, encoding="utf-8-sig")
print("DATOS HISTÓRICOS:", df.shape)
print("CATÁLOGO:", maestro.shape)


VER LOS NOMBRES DE LAS COLUMNAS

In [ ]:

# Mostrar los nombres de todas las columnas
df.columns.tolist()


ELIMINAR COLUMNAS INNECESARIAS

In [ ]:

# Crear una copia para no modificar la base original
df_diagnostico = df.copy()

# Eliminar columnas innecesarias
df_diagnostico = df_diagnostico.drop(columns=[
    'ESTA EN EL NO CANJEO',
    'DENTRO LOS MESES',
    'CUMPLE CAMBIO'
])

# Mostrar las columnas restantes
df_diagnostico.columns.tolist()


TIPOS DE VARIABLES

In [ ]:
# Verificar los tipos de datos
df_diagnostico.dtypes



CONVERTIR MOTIVO A CATEGORÍA

In [ ]:
# Verificar los valores de MOTIVO
df_diagnostico['MOTIVO'].value_counts(dropna=False)


In [ ]:

# Convertir MOTIVO a variable categórica
df_diagnostico['MOTIVO'] = (
    df_diagnostico['MOTIVO']
    .astype('category')
)

# Verificar el nuevo tipo de dato
df_diagnostico['MOTIVO'].dtype


FORMATO DE LAS FECHAS

In [ ]:
# Observar los primeros 10 valores de FECHA DE REVISIÓN
df_diagnostico['FECHA DE REVISIÓN'].head(10)


In [ ]:
# Convertir la columna a tipo fecha
df_diagnostico['FECHA DE REVISIÓN'] = pd.to_datetime(
    df_diagnostico['FECHA DE REVISIÓN'],
    format='%m/%d/%Y',
    errors='coerce'
)


# Comprobar el tipo de dato
print(df_diagnostico['FECHA DE REVISIÓN'].dtype)

# Contar fechas vacías o inválidas
print("Fechas faltantes o inválidas:",
      df_diagnostico['FECHA DE REVISIÓN'].isna().sum())



PARA PUNTO DE VENTA

In [ ]:
# Contar registros por punto de venta
df_diagnostico['PUNTO DE VENTA'].value_counts(dropna=False)



In [ ]:
# Convertir PUNTO DE VENTA a variable categórica
df_diagnostico['PUNTO DE VENTA'] = (
    df_diagnostico['PUNTO DE VENTA']
    .astype('category')
)

# Verificar el tipo de variable
df_diagnostico['PUNTO DE VENTA'].dtype


CANTIDAD

In [ ]:
# Observar los primeros 10 valores de CANTIDAD
df_diagnostico['CANTIDAD'].head(10)


In [ ]:
# Convertir CANTIDAD a tipo numérico
df_diagnostico['CANTIDAD'] = pd.to_numeric(
    df_diagnostico['CANTIDAD'],
    errors='coerce'
).astype('Int64')


In [ ]:
# Verificar tipo de dato
print(df_diagnostico['CANTIDAD'].dtype)

# Contar cantidades faltantes
print("Cantidades faltantes:",
      df_diagnostico['CANTIDAD'].isna().sum())


FECHA

In [ ]:
# Observar los primeros 15 valores de FECHA
df_diagnostico['FECHA'].head(15)


In [ ]:
# Convertir FECHA a tipo fecha
df_diagnostico['FECHA'] = pd.to_datetime(
    df_diagnostico['FECHA'],
    format='%m/%d/%Y',
    errors='coerce'
)


In [ ]:
# Verificar el tipo de dato
print(df_diagnostico['FECHA'].dtype)

# Contar fechas faltantes
print("Fechas faltantes:",
      df_diagnostico['FECHA'].isna().sum())


NOMBRE ESTÁNDAR

In [ ]:
# Convertir NOMBRE ESTÁNDAR a texto
df_diagnostico['NOMBRE ESTÁNDAR'] = (
    df_diagnostico['NOMBRE ESTÁNDAR']
    .astype('string')
    .str.strip()
    .str.upper()
)


In [ ]:
# Verificar el tipo de dato
print(df_diagnostico['NOMBRE ESTÁNDAR'].dtype)

# Mostrar los primeros 10 valores
df_diagnostico['NOMBRE ESTÁNDAR'].head(10)


CÓDIGOS DE BARRAS Y DE PRODUCTO

In [ ]:
# Convertir CÓDIGO DE BARRA a texto
df_diagnostico['CÓDIGO DE BARRA'] = (
    df_diagnostico['CÓDIGO DE BARRA']
    .astype('string')
    .str.strip()
)


In [ ]:
# Convertir CÓDIGO DE PRODUCTO a texto
df_diagnostico['CÓDIGO DE PRODUCTO'] = (
    df_diagnostico['CÓDIGO DE PRODUCTO']
    .astype('string')
    .str.strip()
)


In [ ]:
# Verificar los tipos de todas las variables
df_diagnostico.dtypes


ESTADÍSTICAS GENERALES DE LA BASE DE DATOS

In [ ]:
# Estadísticas generales de la base
print("Total de registros:", len(df_diagnostico))
print("Total de variables:", df_diagnostico.shape[1])

print("Número de sucursales:",
      df_diagnostico['PUNTO DE VENTA'].nunique())

print("Nombres estándar diferentes:",
      df_diagnostico['NOMBRE ESTÁNDAR'].nunique())


DISTRIBUCIÓN DE REGISTROS SEGÚN MOTIVO

In [ ]:
# Contar registros según el motivo
df_diagnostico['MOTIVO'].value_counts()


In [ ]:
# Crear tabla de estadísticas generales
resumen_general = pd.DataFrame({
    'INDICADOR': [
        'Total de registros',
        'Número de variables',
        'Sucursales',
        'Nombres estándar diferentes',
        'Registros vencidos',
        'Registros pinchados'
    ],
    'VALOR': [
        len(df_diagnostico),
        df_diagnostico.shape[1],
        df_diagnostico['PUNTO DE VENTA'].nunique(),
        df_diagnostico['NOMBRE ESTÁNDAR'].nunique(),
        (df_diagnostico['MOTIVO'] == 'VENCIDO').sum(),
        (df_diagnostico['MOTIVO'] == 'PINCHADO').sum()
    ]
})

display(resumen_general)


DEFINIR COLUMNAS OBLIGATORIAS

In [ ]:
# Definir las columnas obligatorias
columnas_obligatorias = [
    'MOTIVO',
    'FECHA DE REVISIÓN',
    'PUNTO DE VENTA',
    'CANTIDAD',
    'NOMBRE ESTÁNDAR',
    'CÓDIGO DE BARRA',
    'CÓDIGO DE PRODUCTO'
]


In [ ]:
# Convertir marcadores de ausencia en valores nulos
df_diagnostico[columnas_obligatorias] = (
    df_diagnostico[columnas_obligatorias]
    .replace(r'^\s*$', pd.NA, regex=True)
    .replace(
        r'(?i)^\s*(no encontrado|nan|null|none)\s*$',
        pd.NA,
        regex=True
    )
)


In [ ]:
# Clasificar registros completos e incompletos
df_diagnostico['ESTADO REGISTRO'] = (
    df_diagnostico[columnas_obligatorias]
    .notna()
    .all(axis=1)
    .map({True: 'COMPLETO', False: 'INCOMPLETO'})
)

# Mostrar resultados
df_diagnostico['ESTADO REGISTRO'].value_counts()


In [ ]:
# Calcular porcentajes de completitud
porcentajes = (
    df_diagnostico['ESTADO REGISTRO']
    .value_counts(normalize=True) * 100
).round(2)

porcentajes


In [ ]:
# Crear tabla de completitud
tabla_completitud = (
    df_diagnostico['ESTADO REGISTRO']
    .value_counts()
    .to_frame('CANTIDAD')
)

# Añadir los porcentajes
tabla_completitud['PORCENTAJE (%)'] = porcentajes

tabla_completitud


In [ ]:
# Crear gráfico de barras
ax = tabla_completitud['PORCENTAJE (%)'].plot(
    kind='bar',
    figsize=(7, 5),
    color=['#2E8B57', '#E57373']
)

# Agregar títulos
plt.title('Nivel de completitud de los registros')
plt.xlabel('Estado del registro')
plt.ylabel('Porcentaje (%)')

# Mostrar porcentajes sobre las barras
for barras in ax.containers:
    ax.bar_label(barras, fmt='%.2f%%', padding=3)

plt.xticks(rotation=0)
plt.ylim(0, 105)
plt.tight_layout()
plt.show()


CUMPLIMIENTO SEGÚN MOTIVO

In [ ]:
# Verificar datos faltantes según motivo
tabla_faltantes = df_diagnostico.groupby(
    'MOTIVO', observed=True
)[['CANTIDAD', 'FECHA']].apply(
    lambda datos: datos.isna().sum()
)

tabla_faltantes


In [ ]:
# Función para evaluar el cumplimiento
def validar_cumplimiento(fila):

    if fila['MOTIVO'] == 'PINCHADO':
        return 'CUMPLE' if pd.notna(fila['CANTIDAD']) else 'NO CUMPLE'

    elif fila['MOTIVO'] == 'VENCIDO':
        return 'CUMPLE' if (
            pd.notna(fila['CANTIDAD']) and
            pd.notna(fila['FECHA'])
        ) else 'NO CUMPLE'

    return 'NO APLICA'


In [ ]:
# Evaluar todos los registros
df_diagnostico['CUMPLIMIENTO SEGÚN MOTIVO'] = (
    df_diagnostico.apply(validar_cumplimiento, axis=1)
)

# Contar los resultados
df_diagnostico['CUMPLIMIENTO SEGÚN MOTIVO'].value_counts()


In [ ]:
# Crear tabla de cumplimiento por motivo
tabla_cumplimiento = pd.crosstab(
    df_diagnostico['MOTIVO'],
    df_diagnostico['CUMPLIMIENTO SEGÚN MOTIVO']
)

tabla_cumplimiento


In [ ]:
# Calcular porcentajes por motivo
tabla_porcentaje = (
    tabla_cumplimiento
    .div(tabla_cumplimiento.sum(axis=1), axis=0)
    * 100
).round(2)

# Mostrar resultados
tabla_porcentaje


In [ ]:
# Crear gráfico de barras agrupadas
ax = tabla_porcentaje.plot(
    kind='bar',
    figsize=(9, 5),
    color=['#2E8B57', '#E57373']
)

# Títulos y ejes
plt.title('Cumplimiento de registros según motivo')
plt.xlabel('Motivo del reporte')
plt.ylabel('Porcentaje (%)')

# Mostrar porcentajes encima de las barras
for barras in ax.containers:
    ax.bar_label(barras, fmt='%.2f%%', padding=3)

# Ajustes del gráfico
plt.xticks(rotation=0)
plt.ylim(0, 110)
plt.legend(title='Estado de cumplimiento')
plt.tight_layout()
plt.show()


COMPORTAMIENTO DE LOS REPORTES

In [ ]:
# Obtener año y semana de cada reporte
iso = df_diagnostico['FECHA DE REVISIÓN'].dt.isocalendar()

# Crear las columnas
df_diagnostico['AÑO'] = iso.year
df_diagnostico['SEMANA'] = iso.week

# Visualizar el resultado
display(
    df_diagnostico[
        ['FECHA DE REVISIÓN', 'AÑO', 'SEMANA']
    ].head(10)
)


In [ ]:
# Identificar el período de análisis
print("Primera fecha:",
      df_diagnostico['FECHA DE REVISIÓN'].min())

print("Última fecha:",
      df_diagnostico['FECHA DE REVISIÓN'].max())

print("Semanas diferentes:",
      df_diagnostico[['AÑO', 'SEMANA']].drop_duplicates().dropna().shape[0])


In [ ]:
# Crear el período semanal
periodo = df_diagnostico['FECHA DE REVISIÓN'].dt.to_period('W-SUN')

# Obtener inicio y fin de semana
df_diagnostico['INICIO SEMANA'] = periodo.dt.start_time
df_diagnostico['FIN SEMANA'] = periodo.dt.end_time.dt.normalize()


In [ ]:
# Mostrar las primeras fechas y sus semanas
display(
    df_diagnostico[
        ['FECHA DE REVISIÓN',
         'AÑO',
         'SEMANA',
         'INICIO SEMANA',
         'FIN SEMANA']
    ].head(10)
)


In [ ]:
# Contar registros por semana y sucursal
reportes_semanales = (
    df_diagnostico
    .groupby(
        ['AÑO', 'SEMANA', 'PUNTO DE VENTA'],
        observed=True
    )
    .size()
    .reset_index(name='NÚMERO DE REGISTROS')
)

display(reportes_semanales.head(20))


In [ ]:
# Lista de sucursales observadas en el conjunto de datos.
# Para medir cumplimiento formal, contrastar con un padrón externo de sucursales obligadas.
sucursales = sorted(df_diagnostico['PUNTO DE VENTA'].dropna().astype(str).unique().tolist())
print('Sucursales con algún registro en el período:', len(sucursales))


In [ ]:
# Crear todas las semanas entre la primera y última fecha
semanas = pd.period_range(
    start=df_diagnostico['FECHA DE REVISIÓN'].min(),
    end=df_diagnostico['FECHA DE REVISIÓN'].max(),
    freq='W-SUN'
)

print("Semanas del período:", len(semanas))


In [ ]:
# Crear todas las combinaciones de semanas y sucursales
esperados = pd.MultiIndex.from_product(
    [semanas, sucursales],
    names=['PERIODO_SEMANA', 'PUNTO DE VENTA']
).to_frame(index=False)

# Mostrar las primeras filas
display(esperados.head(15))


In [ ]:
# Crear período semanal de cada registro
df_diagnostico['PERIODO_SEMANA'] = (
    df_diagnostico['FECHA DE REVISIÓN']
    .dt.to_period('W-SUN')
)


In [ ]:
# Identificar sucursales con registros por semana
reportados = df_diagnostico[
    ['PERIODO_SEMANA', 'PUNTO DE VENTA']
].dropna().drop_duplicates()

# Crear columna de confirmación
reportados['REPORTÓ'] = 'SI'

display(reportados.head(15))


In [ ]:
# Comparar lo esperado con lo reportado
control_semanal = esperados.merge(
    reportados,
    on=['PERIODO_SEMANA', 'PUNTO DE VENTA'],
    how='left'
)


In [ ]:
# Identificar las combinaciones sin registros
control_semanal['REPORTÓ'] = (
    control_semanal['REPORTÓ'].fillna('NO')
)

# Mostrar los primeros resultados
display(control_semanal.head(15))


In [ ]:
# Resumen de cobertura semanal
resumen_semanal = control_semanal.groupby(
    'PERIODO_SEMANA'
).agg(
    SUCURSALES_ESPERADAS=('PUNTO DE VENTA', 'size'),
    SUCURSALES_CON_REGISTROS=('REPORTÓ', lambda x: (x == 'SI').sum())
).reset_index()

display(resumen_semanal)


In [ ]:
# Calcular porcentaje de cobertura
resumen_semanal['COBERTURA (%)'] = (
    resumen_semanal['SUCURSALES_CON_REGISTROS']
    / resumen_semanal['SUCURSALES_ESPERADAS'] * 100
).round(2)

display(resumen_semanal)


In [ ]:
# Crear etiquetas de las semanas
resumen_semanal['SEMANA'] = (
    'S' +
    resumen_semanal['PERIODO_SEMANA']
    .dt.start_time.dt.isocalendar().week.astype(str)
)

# Crear gráfico de líneas
plt.figure(figsize=(12, 5))

plt.plot(
    resumen_semanal['SEMANA'],
    resumen_semanal['COBERTURA (%)'],
    marker='o',
    linewidth=2
)

plt.title('Cobertura semanal de registros por sucursal')
plt.xlabel('Semana')
plt.ylabel('Cobertura (%)')
plt.ylim(0, 110)
plt.grid(True, alpha=0.3)

# Mostrar porcentajes
for x, y in zip(
    resumen_semanal['SEMANA'],
    resumen_semanal['COBERTURA (%)']
):
    plt.annotate(
        f'{y:.1f}%',
        (x, y),
        textcoords='offset points',
        xytext=(0, 8),
        ha='center',
        fontsize=8
    )

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Calcular promedio de cobertura semanal
promedio_cobertura = resumen_semanal['COBERTURA (%)'].mean()

print(f"Cobertura semanal promedio: {promedio_cobertura:.2f}%")


In [ ]:
# Crear gráfico de cobertura semanal
ax = resumen_semanal.plot(
    x='SEMANA',
    y='COBERTURA (%)',
    kind='line',
    marker='o',
    figsize=(13, 6),
    legend=False
)

# Agregar línea del promedio
plt.axhline(
    y=promedio_cobertura,
    color='red',
    linestyle='--',
    label=f'Promedio: {promedio_cobertura:.2f}%'
)

# Mostrar porcentajes sobre cada punto
for i, valor in enumerate(resumen_semanal['COBERTURA (%)']):
    ax.annotate(
        f'{valor:.1f}%',
        (i, valor),
        xytext=(0, 9),
        textcoords='offset points',
        ha='center',
        fontsize=8
    )

# Configuración
plt.title('Cobertura semanal de registros por sucursal')
plt.xlabel('Semana')
plt.ylabel('Cobertura (%)')

plt.xticks(
    range(len(resumen_semanal)),
    resumen_semanal['SEMANA'],
    rotation=45
)

plt.ylim(0, 115)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()



REGISTROS DUPLICADOS EXACTOS

In [ ]:
# Contar registros duplicados exactos
duplicados = df_diagnostico[
    [
        'MOTIVO',
        'FECHA DE REVISIÓN',
        'PUNTO DE VENTA',
        'PRODUCTO',
        'CANTIDAD',
        'FECHA',
        'NOMBRE ESTÁNDAR',
        'CÓDIGO DE BARRA',
        'CÓDIGO DE PRODUCTO'
    ]
].duplicated().sum()

print("Registros duplicados exactos:", duplicados)


In [ ]:
# Revisar cuántas filas participan en grupos duplicados, sin mostrar datos privados
columnas_duplicados = ['MOTIVO', 'FECHA DE REVISIÓN', 'PUNTO DE VENTA', 'PRODUCTO', 'CANTIDAD', 'FECHA', 'NOMBRE ESTÁNDAR', 'CÓDIGO DE BARRA', 'CÓDIGO DE PRODUCTO']
registros_duplicados = df_diagnostico[df_diagnostico[columnas_duplicados].duplicated(keep=False)]
print('Filas en grupos duplicados:', len(registros_duplicados))


In [ ]:
# Recuento de duplicados adicionales
print('Duplicados exactos adicionales:', duplicados)


VALORES FALTANTE POR VARIABLE

In [ ]:
# Contar valores faltantes por columna obligatoria
faltantes = (
    df_diagnostico[columnas_obligatorias]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(faltantes)


In [ ]:
# Crear gráfico de valores faltantes
ax = faltantes.plot(
    kind='bar',
    figsize=(10, 5),
    color='#4472C4'
)

plt.title('Valores faltantes por variable')
plt.xlabel('Variable')
plt.ylabel('Número de valores faltantes')

# Mostrar cantidades encima de las barras
for barras in ax.containers:
    ax.bar_label(barras, fmt='%d', padding=3)

plt.xticks(rotation=35, ha='right')
plt.ylim(0, 20)
plt.tight_layout()
plt.show()
